<a href="https://colab.research.google.com/github/louistrue/learn-ifc-bfh25-D/blob/main/MVP-Massivbau%2016.12.25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ifcopenshell pandas openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 MB 15.7 MB/s eta 0:00:00


In [12]:
import requests
import ifcopenshell
import pandas as pd
from ifcopenshell.util.element import get_psets

# ----------------------------------
# IFC von GitHub herunterladen
# ----------------------------------
url = "https://raw.githubusercontent.com/louistrue/learn-ifc-bfh25-D/main/Modelle/BFH-25/2379_230926_V1033_BA_41_SCA_300B.ifc"
local_ifc = "modell.ifc"

r = requests.get(url)
r.raise_for_status()

with open(local_ifc, "wb") as f:
    f.write(r.content)

# ----------------------------------
# IFC öffnen
# ----------------------------------
model = ifcopenshell.open(local_ifc)

walls = model.by_type("IfcWall") + model.by_type("IfcWallStandardCase")

rows = []

for wall in walls:
    psets = get_psets(wall)

    betontyp = psets.get("wg_SUB", {}).get("Betontyp")
    qto = psets.get("Qto_WallBaseQuantities", {})

    rows.append({
    "GlobalId": wall.GlobalId,
    "Betontyp": betontyp,
    "eBKP_H_Beschrieb": psets.get("wg_SUB", {}).get("eBKP H Beschrieb"),
    "Wandhoehe_m": qto.get("Height"),
    "Volumen_m3": qto.get("NetVolume"),
    "Flaeche_m2": qto.get("NetSideArea")
})

df = pd.DataFrame(rows)
df.to_excel("Auswertung.xlsx", index=False)

print("Excel-Auswertung erstellt.")



Excel-Auswertung erstellt.


In [15]:
# -------------------------------
# Mapping Betontyp → NPK
# -------------------------------
BETONTYP_TO_NPK = {
    "Betontyp 01": "NPK C",
    "Betontyp 02": "NPK B"
}

df["NPK"] = df["Betontyp"].map(BETONTYP_TO_NPK)

# -------------------------------
# Filter Wandhöhe
# -------------------------------
df = df[df["Wandhoehe_m"] >= 2.50]

# -------------------------------
# In bestehende Excel schreiben
# -------------------------------
with pd.ExcelWriter(
    "Datenmappe NPK.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    df[df["NPK"] == "NPK C"].to_excel(
        writer,
        sheet_name="NPK_C",
        index=False
    )

    df[df["NPK"] == "NPK B"].to_excel(
        writer,
        sheet_name="NPK_B",
        index=False
    )


In [17]:
# df muss aus Ihrer IFC-Auswertung stammen
# Beispiel: df = pd.DataFrame(rows) aus Ihrem IFC-Skript

# Mapping Betontyp -> NPK
BETONTYP_TO_NPK = {
    "Betontyp 01": "NPK C",
    "Betontyp 02": "NPK B"
}

df["NPK"] = df["Betontyp"].map(BETONTYP_TO_NPK)

# Wandhöhe filtern (z.B. >= 2.50 m)
df_filtered = df[df["Wandhoehe_m"] >= 2.50]

# Prüfen, ob df_filtered Daten enthält
print(f"Anzahl der ausgewerteten Wände nach Filter: {len(df_filtered)}")
if df_filtered.empty:
    raise ValueError("Keine Wände erfüllen das Wandhöhenkriterium.")


Anzahl der ausgewerteten Wände nach Filter: 97


In [20]:
import requests
import ifcopenshell
import pandas as pd
from ifcopenshell.util.element import get_psets

# -----------------------------
# 1. IFC von GitHub laden
# -----------------------------
url = "https://raw.githubusercontent.com/louistrue/learn-ifc-bfh25-D/main/Modelle/BFH-25/2379_230926_V1033_BA_41_SCA_300B.ifc"
local_ifc = "modell.ifc"

r = requests.get(url)
r.raise_for_status()
with open(local_ifc, "wb") as f:
    f.write(r.content)

# -----------------------------
# 2. IFC öffnen und Wände auslesen
# -----------------------------
model = ifcopenshell.open(local_ifc)
walls = model.by_type("IfcWall") + model.by_type("IfcWallStandardCase")

rows = []
for wall in walls:
    psets = get_psets(wall)
    wg_sub = psets.get("wg_SUB", {})
    betontyp = wg_sub.get("Betontyp")
    qto = psets.get("Qto_WallBaseQuantities", {})
    hoehe = qto.get("Height")
    volumen = qto.get("NetVolume")
    flaeche = qto.get("NetSideArea")

    rows.append({
        "GlobalId": wall.GlobalId,
        "Betontyp": betontyp,
        "Wandhoehe_m": hoehe,
        "Volumen_m3": volumen,
        "Flaeche_m2": flaeche
    })

df = pd.DataFrame(rows).dropna(subset=["Wandhoehe_m", "Volumen_m3"])

# -----------------------------
# 3. Betontyp → NPK
# -----------------------------
BETONTYP_TO_NPK = {
    "Betontyp 01": "NPK C",
    "Betontyp 02": "NPK B",
    "Betontyp 03": "NPK A"
}
df["NPK"] = df["Betontyp"].map(BETONTYP_TO_NPK)

# -----------------------------
# 4. Wandhöhe-Kategorien
# -----------------------------
def hoehe_kategorie(h):
    if h <= 1.5:
        return "≤1.5m"
    elif 1.51 <= h <= 1.99:
        return "1.51-1.99m"
    elif 2.0 <= h <= 3.0:
        return "2.0-3.0m"
    else:
        return ">3.0m"

df["Hoehe_Kategorie"] = df["Wandhoehe_m"].apply(hoehe_kategorie)

# -----------------------------
# 5. Gruppierte Auswertung
# -----------------------------
df_grouped = df.groupby(["NPK", "Hoehe_Kategorie"]).agg(
    Anzahl_Waende=("GlobalId", "count"),
    Gesamtvolumen_m3=("Volumen_m3", "sum"),
    Gesamtflaeche_m2=("Flaeche_m2", "sum")
).reset_index()

# -----------------------------
# 6. Excel-Ausgabe
# -----------------------------
output_path = "NPK_Waende_Auswertung.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Einzelwände
    df.to_excel(writer, sheet_name="Einzelwaende", index=False)
    # Gruppierte Übersicht
    df_grouped.to_excel(writer, sheet_name="Auswertung_NPK_Hoehe", index=False)

print(f"Neue Excel-Datei erstellt: {output_path}")


Exception ignored in: <function ZipFile.__del__ at 0x78528485cc20>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


Neue Excel-Datei erstellt: NPK_Waende_Auswertung.xlsx
